# 06g — Inventory Simulation

**Purpose:** Convert forecasts/policies into reorder decisions and simulate business outcomes
on the Fold 2 val window. Section 1 builds the one piece that's still missing — a
`reorder_point`/`order_qty` table for Smooth/Erratic — and merges it with Lumpy/Intermittent
(06f) into a single table covering all SKUs. Section 2 (next) runs `simulate_inventory()`
against it.

**Inputs:**
- `unified_lumpy_intermittent_fold2.parquet` — 06f, Lumpy + Intermittent, already final
- `conformal_residuals_fold2.pkl` — 06e, per-SKU residuals + locked service level(s) for Smooth/Erratic
- `tweedie_optimized_fold2.txt` + `features_train_v2.parquet` — 06d/04b, to regenerate point forecasts (see gap note below)
- `sku_regimes_fold2.parquet` — 06c, routing + regime labels

**Outputs:**
- `final_reorder_params_fold2.parquet` — all ~30,490 SKUs, one schema, ready for Section 2

**Two gaps vs. the plan spec, both handled below:**
1. `new_plan.md`'s Pre-flight #1 assumes `tweedie_optimized_predictions_fold2.parquet` exists
   as an input. It doesn't — 06e's save cell (Section 5) persisted `sku_residuals` and run
   metadata only, never the per-row `yhat` frame it built in memory. Point forecasts are
   regenerated below using 06e Section 1's exact logic, with a row-count sanity check against
   06e's own residual counts to confirm the regenerated population matches what the residuals
   were calibrated against.
2. **Service level is regime-specific, not one locked value.** 06e Section 4 (cell 8) picks a
   *different* service level for Smooth vs. Erratic via an elbow-detection heuristic on
   interval width — `conformal['default_service_level']` is a dict (`{'smooth': 'qXX',
   'erratic': 'qXX'}`), not the single flat 0.80 that Lumpy/Intermittent locked to. The first
   draft of this notebook assumed a scalar and crashed on it. Fixed below by threading the
   per-regime level through the loop, and by adding an explicit `service_level` column to the
   final schema (0.80 for Lumpy/Intermittent, per-regime for Smooth/Erratic) so Section 5 can
   report it honestly instead of implying one level applies everywhere.

---

## Section 1 — Reorder Parameter Computation (Smooth/Erratic) + Three-Way Merge


In [3]:
# ── Setup — load 06e's conformal output, regenerate the point forecasts it never saved ──
import numpy as np
import pandas as pd
import lightgbm as lgb
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

PROCESSED_DIR    = '../data/processed'
CALIBRATION_DIR  = f'{PROCESSED_DIR}/calibration'
SEGMENTATION_DIR = f'{PROCESSED_DIR}/segmentation'
MODELS_DIR       = f'{PROCESSED_DIR}/models'
PREDICTIONS_DIR  = f'{PROCESSED_DIR}/predictions'

LEAD_TIME_DAYS = 7   # same lead time 06f Section 6/7 locked for Lumpy + Intermittent

with open(f'{CALIBRATION_DIR}/conformal_residuals_fold2.pkl', 'rb') as f:
    conformal = pickle.load(f)

sku_residuals = conformal['sku_residuals']
# NOT a single float -- 06e Section 4 picks a service level PER REGIME (elbow-detection on
# interval width), e.g. {'smooth': 'q80', 'erratic': 'q90'}. Converted to a fraction below.
SERVICE_LEVEL_BY_REGIME = conformal['default_service_level']
RESIDUAL_UNIT = conformal['residual_unit']
WINNER_MODEL  = conformal['winner_model']

def service_level_frac(regime):
    """'q80' -> 0.80. Falls back to 0.80 if a regime is missing from the dict."""
    label = SERVICE_LEVEL_BY_REGIME.get(regime, 'q80')
    return int(label[1:]) / 100

print(f'Loaded conformal residuals for {len(sku_residuals):,} SKUs.')
print(f'Residual unit: {RESIDUAL_UNIT}')
print(f'Locked service levels by regime: {SERVICE_LEVEL_BY_REGIME}')

# SANITY CHECK, not a blocker: 06e Section 1 filtered to Tweedie-routed SKUs only (~9,196
# expected). If len(sku_residuals) above is closer to the FULL SKU population (~30,490),
# 06e's scope may have changed since last reviewed -- worth confirming on your end. The loop
# below only ever iterates `tweedie_skus` regardless, so this can't corrupt this notebook's
# output, but it's a signal something upstream may be worth a second look.

# Which 06d experiments trained on a 7-day-forward-sum target vs. next-day -- same check
# 06e Section 1 used, kept here so the branch below can't silently drift from what the
# residuals were actually calibrated against.
SEVEN_DAY_TARGET_EXPERIMENTS = {'experiment_a', 'experiment_c'}
TARGET_IS_7DAY = WINNER_MODEL in SEVEN_DAY_TARGET_EXPERIMENTS
assert (RESIDUAL_UNIT == 'expected demand, rolling next-7-days') == TARGET_IS_7DAY, (
    'Winner model / residual unit mismatch vs. 06e -- regenerating yhat below would use the '
    'wrong native-space branch. Stop and check 06e Section 1 before proceeding.'
)

FOLD2_VAL_START = '2014-02-01'
CALIBRATION_END = '2014-08-31'   # same calibration window the residuals were built on

model = lgb.Booster(model_file=f'{MODELS_DIR}/tweedie_optimized_fold2.txt')
sku_regimes = pd.read_parquet(f'{SEGMENTATION_DIR}/sku_regimes_fold2.parquet')
tweedie_skus = set(sku_regimes.loc[sku_regimes['routing'] == 'tweedie', 'id'])
regime_map = sku_regimes.set_index('id')['regime'].to_dict()

print(f'Tweedie-routed SKUs (Smooth+Erratic, from 06c routing): {len(tweedie_skus):,}')

train_full = pd.read_parquet(f'{PROCESSED_DIR}/features/features_train_v2.parquet')
train_full['date'] = pd.to_datetime(train_full['date'])
with open(f'{PROCESSED_DIR}/features/feature_cols_v2.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

mask_se    = train_full['id'].isin(tweedie_skus)
mask_calib = (train_full['date'] >= FOLD2_VAL_START) & (train_full['date'] <= CALIBRATION_END)
calib_se = train_full[mask_se & mask_calib].sort_values(['id', 'date']).copy()
calib_se['yhat'] = np.maximum(model.predict(calib_se[feature_cols].values), 0)

print(f"Regenerated yhat for {calib_se['id'].nunique():,} Smooth/Erratic SKUs "
      f'over the calibration window ({len(calib_se):,} rows).')

# Sanity check: regenerated population should match 06e's residual counts row-for-row for
# the Tweedie-routed SKUs specifically -- same window, same SKUs, same model. A mismatch
# here means these yhat values aren't the same population the residuals were calibrated
# against, and nothing below can be trusted.
residual_counts = pd.Series({k: len(v) for k, v in sku_residuals.items() if k in tweedie_skus})
row_counts = calib_se.groupby('id').size()
common = residual_counts.index.intersection(row_counts.index)
mismatch = (residual_counts.loc[common] != row_counts.loc[common]).sum()
print(f'Tweedie SKUs with mismatched row/residual counts vs. 06e: {mismatch:,} / {len(common):,}')

missing_from_calib = sorted(tweedie_skus - set(row_counts.index))
print(f'Tweedie-routed SKUs with zero rows in the calibration window: {len(missing_from_calib):,}')
if missing_from_calib:
    print('  These have no history to forecast from and will be excluded below -- see Findings.')


Loaded conformal residuals for 30,490 SKUs.
Residual unit: expected demand, next-day
Locked service levels by regime: {'smooth': 'q80', 'erratic': 'q80'}
Tweedie-routed SKUs (Smooth+Erratic, from 06c routing): 9,219
Regenerated yhat for 9,171 Smooth/Erratic SKUs over the calibration window (1,806,148 rows).
Tweedie SKUs with mismatched row/residual counts vs. 06e: 0 / 9,219
Tweedie-routed SKUs with zero rows in the calibration window: 0


In [4]:
# ── compute_reorder_params — Smooth/Erratic only (Lumpy/Intermittent read directly in the ──
# merge cell below, per Pre-flight #1; nothing here touches those two regimes)
MIN_RESIDUALS_PER_SKU = 5   # same floor 06e Section 2 used for a stable per-SKU quantile

point_forecast = calib_se.groupby('id')['yhat'].mean()   # "typical demand," same framing as
                                                          # Lumpy's avg_weekly / Intermittent's
                                                          # mean_lead_time_demand -- a single
                                                          # static value per SKU, not the most
                                                          # recent day's forecast (Pre-flight #3
                                                          # needs a fixed value, not a moving one)

# Pooled fallback for SKUs with too few calibration residuals for their own stable quantile.
# Built PER REGIME now, not globally pooled -- Smooth and Erratic use different service
# levels, so a single global pooled quantile would apply the wrong level to half the fallback.
pooled_residuals_by_regime = {}
for regime in ('smooth', 'erratic'):
    ids_in_regime = {sid for sid, r in regime_map.items() if r == regime}
    pooled = np.concatenate([
        np.asarray(v) for sid, v in sku_residuals.items() if sid in ids_in_regime
    ]) if ids_in_regime else np.array([0.0])
    pooled_residuals_by_regime[regime] = pooled

rows = []
n_negative_q80 = 0
n_nan_forecast = 0
nan_forecast_ids = []
for sku_id in sorted(tweedie_skus):
    if sku_id not in point_forecast.index:
        continue  # no calibration-window history -- flagged above, excluded here

    regime = regime_map[sku_id]
    sl = service_level_frac(regime)   # regime-specific, not a flat 0.80

    pf_native = point_forecast[sku_id]
    residuals = sku_residuals.get(sku_id, [])
    low_conf  = len(residuals) < MIN_RESIDUALS_PER_SKU
    if low_conf:
        resid_q = np.percentile(pooled_residuals_by_regime.get(regime, [0.0]), sl * 100)
    else:
        resid_q = np.percentile(residuals, sl * 100)
    if resid_q < 0:
        n_negative_q80 += 1
    resid_q = max(resid_q, 0.0)   # a SKU that mostly over-forecasts gets no negative "safety
                                   # debt" -- floored the same way Intermittent's reactive-only
                                   # cohort was in 06f, not left as-is

    # NOTE: residual = target - yhat already, so `resid_q` IS the interval half-width
    # (equivalent to the plan's `q_upper - point_forecast`) -- no second subtraction needed.
    if TARGET_IS_7DAY:
        # Model already predicts a native 7-day-forward sum -- no daily->lead-time scaling.
        expected_lead_time_demand = pf_native
        safety_stock = resid_q
    else:
        # Model predicts next-day units -- scaling to the 7-day lead time via sqrt(7).
        # UNVALIDATED (Pre-flight #4): assumes i.i.d. daily forecast errors. 06f Section 6
        # found the opposite for Intermittent (real demand clusters in time, ~2.9x more
        # lead-time variance than an independent-day model implies). Smooth/Erratic demand is
        # more regular and may hold up better, but that hasn't been checked with the same
        # rolling-origin coverage test yet -- do not treat these reorder points as trustworthy
        # until that runs.
        expected_lead_time_demand = pf_native * LEAD_TIME_DAYS
        safety_stock = resid_q * np.sqrt(LEAD_TIME_DAYS)

    # GUARD: Python's built-in max() does NOT floor NaN -- max(nan, 1.0) returns nan, because
    # every comparison against nan is False. If the point forecast came out NaN/inf (e.g. a
    # SKU with too little lag-feature history in the calibration window), silently taking
    # max(nan, 1.0) would let a NaN slip into order_qty and fail the assert below with no
    # visibility into which SKU or why. Caught explicitly instead: treated as low-confidence/
    # fallback (same as the zero-history cohort), not guessed at.
    if not (np.isfinite(expected_lead_time_demand) and np.isfinite(safety_stock)):
        n_nan_forecast += 1
        nan_forecast_ids.append(sku_id)
        low_conf = True
        fallback_required = True
        expected_lead_time_demand = 0.0
        safety_stock = 0.0
        reorder_point = 0.0
        order_qty = 1.0
    else:
        fallback_required = low_conf
        reorder_point = expected_lead_time_demand + safety_stock
        order_qty = max(expected_lead_time_demand, 1.0)   # never order zero -- same floor logic as 06f

    rows.append({
        'id':                        sku_id,
        'regime':                    regime,
        'method':                    'conformal_tweedie',
        'expected_lead_time_demand': expected_lead_time_demand,
        'reorder_point':             reorder_point,
        'safety_buffer':             safety_stock,
        'order_qty':                 order_qty,
        'low_confidence':            low_conf,
        'fallback_required':         fallback_required,
        'sqrt_scaling_unvalidated':  not TARGET_IS_7DAY,
        'service_level':             sl,
    })

smooth_erratic_unified = pd.DataFrame(rows)

print(f'SKUs with NaN/inf point forecast (guarded, forced to fallback): {n_nan_forecast:,}')
if nan_forecast_ids:
    print(f'  Sample IDs: {nan_forecast_ids[:10]}')
    print('  Worth checking whether these share a cause (e.g. launched too recently for full')
    print('  lag-feature history in the calibration window) before trusting the fallback value.')

assert smooth_erratic_unified['order_qty'].notna().all() and (smooth_erratic_unified['order_qty'] > 0).all(), (
    'BUG: some Smooth/Erratic SKU has order_qty <= 0 -- would never reorder in simulation.'
)

print(f'Smooth/Erratic reorder table: {len(smooth_erratic_unified):,} SKUs.')
print(f'Low-confidence (<{MIN_RESIDUALS_PER_SKU} calib residuals OR NaN forecast): '
      f"{smooth_erratic_unified['low_confidence'].sum():,}")
print(f'SKUs where the raw q_level residual was negative before flooring: {n_negative_q80:,} '
      f'(over-forecasting cohort -- reactive-only, same shape as Intermittent\'s cohort in 06f)')
print()
print(smooth_erratic_unified.groupby('regime')[
    ['expected_lead_time_demand', 'reorder_point', 'safety_buffer', 'order_qty', 'service_level']
].describe().round(2))


SKUs with NaN/inf point forecast (guarded, forced to fallback): 48
  Sample IDs: ['FOODS_1_019_CA_1_validation', 'FOODS_1_019_CA_2_validation', 'FOODS_1_019_CA_4_validation', 'FOODS_1_019_TX_1_validation', 'FOODS_1_019_TX_2_validation', 'FOODS_1_019_TX_3_validation', 'FOODS_1_049_WI_3_validation', 'FOODS_2_103_CA_4_validation', 'FOODS_2_358_TX_3_validation', 'FOODS_3_096_WI_3_validation']
  Worth checking whether these share a cause (e.g. launched too recently for full
  lag-feature history in the calibration window) before trusting the fallback value.
Smooth/Erratic reorder table: 9,219 SKUs.
Low-confidence (<5 calib residuals OR NaN forecast): 48
SKUs where the raw q_level residual was negative before flooring: 655 (over-forecasting cohort -- reactive-only, same shape as Intermittent's cohort in 06f)

        expected_lead_time_demand                                        \
                            count   mean    std  min   25%   50%    75%   
regime                             

In [5]:
# ── Cadence check + three-way merge ──────────────────────────────────────────────────────
# Mirrors 06f Section 7's cadence assert before it concatenated Lumpy + Intermittent --
# confirms Smooth/Erratic's regenerated 7-day lead-time figures and Lumpy/Intermittent's
# weekly figures are actually on the same clock before treating them as comparable.
assert LEAD_TIME_DAYS == 7, (
    'Smooth/Erratic reorder points above assume a 7-day lead time. Lumpy (06f Section 4, '
    'lead_time_weeks=1) and Intermittent (06f Section 6, LEAD_TIME_DAYS=7) both assumed the '
    'same -- if LEAD_TIME_DAYS changes here, the three regimes are no longer on the same '
    'clock and cannot be merged as-is.'
)

lumpy_intermittent = pd.read_parquet(f'{PREDICTIONS_DIR}/unified_lumpy_intermittent_fold2.parquet')
lumpy_intermittent['sqrt_scaling_unvalidated'] = False   # policy / block-bootstrap methods --
                                                          # no i.i.d.-daily-error assumption involved
lumpy_intermittent['service_level'] = 0.80               # locked flat value, per 06f Section 6

final_reorder_params = pd.concat([lumpy_intermittent, smooth_erratic_unified], ignore_index=True)

expected_total = sku_regimes['id'].nunique()
actual_total   = len(final_reorder_params)
print(f'Merged table: {actual_total:,} SKUs vs. {expected_total:,} in 06c\'s full routing '
      f'({expected_total - actual_total:,} short -- should equal the zero-calibration-history '
      f'count printed above, if any).')

assert final_reorder_params['order_qty'].notna().all() and (final_reorder_params['order_qty'] > 0).all(), (
    'order_qty must be populated and positive for every SKU across all three regimes before '
    'Section 2 can run simulate_inventory().'
)

print()
print(final_reorder_params['regime'].value_counts())
print()
print('Service level in effect, by regime:')
print(final_reorder_params.groupby('regime')['service_level'].agg(['min', 'max', 'mean']).round(2))
print()
print(f'Low-confidence / fallback SKUs by regime (Smooth/Erratic only -- Lumpy/Intermittent '
      f'have their own flags from 06f):')
print(final_reorder_params.groupby('regime')[['low_confidence', 'fallback_required']].mean().round(3) * 100)

os.makedirs(PREDICTIONS_DIR, exist_ok=True)
final_reorder_params.to_parquet(f'{PREDICTIONS_DIR}/final_reorder_params_fold2.parquet')
print(f'\n✓ final_reorder_params_fold2.parquet saved ({actual_total:,} SKUs, all regimes).')
print('REMINDER: Smooth/Erratic reorder points carry an unvalidated sqrt(lead_time) daily-error')
print('assumption (Pre-flight #4, sqrt_scaling_unvalidated=True) -- run the rolling-origin')
print('coverage check before Section 2 trusts them in the simulation.')


Merged table: 30,490 SKUs vs. 30,490 in 06c's full routing (0 short -- should equal the zero-calibration-history count printed above, if any).

regime
intermittent    14268
smooth           8389
lumpy            7003
erratic           830
Name: count, dtype: int64

Service level in effect, by regime:
              min  max  mean
regime                      
erratic       0.8  0.8   0.8
intermittent  0.8  0.8   0.8
lumpy         0.8  0.8   0.8
smooth        0.8  0.8   0.8

Low-confidence / fallback SKUs by regime (Smooth/Erratic only -- Lumpy/Intermittent have their own flags from 06f):
              low_confidence  fallback_required
regime                                         
erratic                  0.4                0.4
intermittent             0.0                0.0
lumpy                   59.8               59.8
smooth                   0.5                0.5

✓ final_reorder_params_fold2.parquet saved (30,490 SKUs, all regimes).
REMINDER: Smooth/Erratic reorder points carry a

### Section 1 Findings — Reorder Parameter Computation + Three-Way Merge

**Coverage:** [RERUN NEEDED: paste output] SKUs merged (7,003 Lumpy + 14,268 Intermittent +
[RERUN NEEDED] Smooth/Erratic) against 06c's full routing of [RERUN NEEDED] SKUs.

**Gap closed:** `tweedie_optimized_predictions_fold2.parquet`, which the original plan assumed
as an input, was never saved by 06e. Point forecasts were regenerated directly from the 06d
winner model + features, using the identical native-space logic 06e Section 1 used, and
cross-checked row-for-row against 06e's own residual counts to confirm the regenerated
population matches what the residuals were actually calibrated against ([RERUN NEEDED:
mismatch count] mismatches found).

**Service level is regime-specific, not flat:** 06e Section 4 picked a different level for
Smooth vs. Erratic via an elbow-detection heuristic on interval width, not the single 0.80
Lumpy/Intermittent locked to. [RERUN NEEDED: paste the actual `SERVICE_LEVEL_BY_REGIME` dict
and note whether Smooth/Erratic ended up above, below, or matching 0.80.]

**SKU population check:** `conformal_residuals_fold2.pkl` loaded with [RERUN NEEDED: paste
`len(sku_residuals)`] entries — flagged as worth confirming against the ~9,196 Tweedie-routed
population 06e Section 1 originally scoped to, since this run reported 30,490 (closer to the
full four-regime SKU count). Did not block this notebook (the loop only ever iterates
`tweedie_skus`), but worth a follow-up check on whether 06e's scope changed.

**Zero-history SKUs:** [RERUN NEEDED: count] Tweedie-routed SKUs had no rows in the
calibration window and were excluded from the reorder table entirely, rather than assigned a
fabricated forecast.

**Low-confidence cohort:** [RERUN NEEDED: count] Smooth/Erratic SKUs had fewer than 5
calibration-window residuals and fell back to their regime's pooled quantile rather than an
unstable per-SKU estimate.

**Still open before Section 2 can trust these numbers (Pre-flight #4):** every Smooth/Erratic
`reorder_point` above used `sqrt(7)` to scale a next-day residual quantile into a 7-day
lead-time safety stock — an untested i.i.d.-daily-error assumption, flagged per-row via
`sqrt_scaling_unvalidated=True`. This needs the same rolling-origin coverage check 06f
Section 6 ran for Intermittent before Section 2's simulation results can be trusted for this
regime specifically.

**Output:** `final_reorder_params_fold2.parquet`, schema: `id, regime, method,
expected_lead_time_demand, reorder_point, safety_buffer, order_qty, low_confidence,
fallback_required, sqrt_scaling_unvalidated, service_level` — all SKUs, one table, ready for
Section 2's `simulate_inventory()`.
